# 可选实验：Multiple Variable Linear Regression

在本实验中，你将扩展之前的数据结构和函数，使其支持多个 feature（特征）。需要更新的函数较多，因此实验看起来较长，但每个函数都只需在原有基础上稍作调整，可以很快完成回顾。

## 目录

- [1.1 目标](#toc_15456_1.1)
- [1.2 工具](#toc_15456_1.2)
- [1.3 符号说明](#toc_15456_1.3)
- [2 问题描述](#toc_15456_2)
  - [2.1 存储样本的 Matrix X](#toc_15456_2.1)
  - [2.2 参数 Vector w 与 Scalar b](#toc_15456_2.2)
- [3 多变量模型的预测](#toc_15456_3)
  - [3.1 逐元素计算单个预测值](#toc_15456_3.1)
  - [3.2 使用 Vector 计算单个预测值](#toc_15456_3.2)
- [4 计算多变量模型的 Cost](#toc_15456_4)
- [5 多变量 Gradient Descent](#toc_15456_5)
  - [5.1 计算多变量 Gradient](#toc_15456_5.1)
  - [5.2 多变量 Gradient Descent](#toc_15456_5.2)
- [6 恭喜完成实验](#toc_15456_6)


<a name="toc_15456_1.1"></a>
## 1.1 目标

扩展 regression（回归）模型的相关函数，使其支持多个 feature：

- 扩展数据结构以支持多个 feature。
- 重写 prediction、cost 和 gradient 的计算函数。
- 使用 NumPy 的 `np.dot` 实现 vectorization，简化代码并提高运行速度。


<a name="toc_15456_1.2"></a>
## 1.2 工具

本实验将使用：

- NumPy：常用的科学计算库。
- Matplotlib：常用的数据绘图库。


In [ ]:
import copy, math
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
np.set_printoptions(precision=2)  # reduced display precision on numpy arrays

<a name="toc_15456_1.3"></a>
## 1.3 符号说明

下面汇总了本实验中使用的符号，并对多 feature 的情况进行了补充。

| 符号 | 说明 | Python（如适用） |
| :--- | :--- | :--- |
| $a$ | scalar（标量），不使用粗体 | |
| $\mathbf{a}$ | vector（向量），使用粗体 | |
| $\mathbf{A}$ | matrix（矩阵），使用大写粗体 | |
| **Regression** | | |
| $\mathbf{X}$ | 训练样本 matrix | `X_train` |
| $\mathbf{y}$ | 训练样本的 target（目标值） | `y_train` |
| $\mathbf{x}^{(i)}$, $y^{(i)}$ | 第 $i$ 个训练样本 | `X[i]`, `y[i]` |
| $m$ | 训练样本数 | `m` |
| $n$ | 每个样本的 feature 数量 | `n` |
| $\mathbf{w}$ | 模型参数：weight（权重） | `w` |
| $b$ | 模型参数：bias（偏置） | `b` |
| $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ | 参数为 $\mathbf{w},b$ 时，模型对 $\mathbf{x}^{(i)}$ 的预测：$f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\mathbf{w}\cdot\mathbf{x}^{(i)}+b$ | `f_wb` |


<a name="toc_15456_2"></a>
## 2 问题描述

本实验继续以房价预测为例。训练数据集包含 3 个样本，每个样本有 4 个 feature：面积、卧室数量、楼层数和房龄，如下表所示。注意，与更早的实验不同，这里的面积以 sqft 为单位，而不是 1000 sqft。这会带来一个问题，你将在下一个实验中解决它！

| 面积（sqft） | 卧室数量 | 楼层数 | 房龄（年） | 价格（千美元） |
| --- | --- | --- | --- | --- |
| 2104 | 5 | 1 | 45 | 460 |
| 1416 | 3 | 2 | 40 | 232 |
| 852 | 2 | 1 | 35 | 178 |

你将利用这些数据构建 linear regression 模型，预测其他房屋的价格，例如一套面积为 1200 sqft、有 3 间卧室、1 层楼、房龄为 40 年的房屋。

请运行下面的代码单元，创建变量 `X_train` 和 `y_train`。


In [ ]:
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

<a name="toc_15456_2.1"></a>
### 2.1 存储样本的 Matrix X

与上表类似，样本存储在 NumPy matrix `X_train` 中，每一行代表一个样本。当训练数据有 $m$ 个样本、每个样本有 $n$ 个 feature 时，$\mathbf{X}$ 的 shape 为 $(m,n)$，即 $m$ 行、$n$ 列。本例中 $m=3$、$n=4$。

$$
\mathbf{X} =
\begin{pmatrix}
x^{(0)}_0 & x^{(0)}_1 & \cdots & x^{(0)}_{n-1} \\
x^{(1)}_0 & x^{(1)}_1 & \cdots & x^{(1)}_{n-1} \\
\vdots & \vdots & \ddots & \vdots \\
x^{(m-1)}_0 & x^{(m-1)}_1 & \cdots & x^{(m-1)}_{n-1}
\end{pmatrix}
$$

符号说明：

- $\mathbf{x}^{(i)}$ 是包含第 $i$ 个样本的 vector，即 $\mathbf{x}^{(i)}=(x^{(i)}_0,x^{(i)}_1,\cdots,x^{(i)}_{n-1})$。
- $x^{(i)}_j$ 是第 $i$ 个样本的第 $j$ 个元素。带括号的上标表示样本编号，下标表示元素编号。

下面显示输入数据。


In [ ]:
# data is stored in numpy array/matrix
print(f"X Shape: {X_train.shape}, X Type:{type(X_train)})")
print(X_train)
print(f"y Shape: {y_train.shape}, y Type:{type(y_train)})")
print(y_train)

<a name="toc_15456_2.2"></a>
### 2.2 参数 Vector w 与 Scalar b

$\mathbf{w}$ 是包含 $n$ 个元素的 vector，每个元素对应一个 feature 的参数。本数据集中 $n=4$。在数学表示中，通常将其写成 column vector（列向量）：

$$
\mathbf{w} =
\begin{pmatrix}
w_0 \\
w_1 \\
\vdots \\
w_{n-1}
\end{pmatrix}
$$

$b$ 是一个 scalar 参数。


为方便演示，下面为 $\mathbf{w}$ 和 $b$ 赋予一组接近最优解的初始值。其中，$\mathbf{w}$ 是一个 1-D NumPy vector。


In [ ]:
b_init = 785.1811367994083
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618])
print(f"w_init shape: {w_init.shape}, b_init type: {type(b_init)}")

<a name="toc_15456_3"></a>
## 3 多变量模型的预测

多变量模型的预测由下面的 linear model 给出：

$$
f_{\mathbf{w},b}(\mathbf{x})=w_0x_0+w_1x_1+\cdots+w_{n-1}x_{n-1}+b \tag{1}
$$

也可以写成 vector 形式：

$$
f_{\mathbf{w},b}(\mathbf{x})=\mathbf{w}\cdot\mathbf{x}+b \tag{2}
$$

其中，$\cdot$ 表示 vector 的 dot product（点积）。

为了说明 dot product 的作用，我们将分别使用公式 (1) 和 (2) 实现预测。


<a name="toc_15456_3.1"></a>
### 3.1 逐元素计算单个预测值

之前的预测方法是将一个 feature 值乘以一个参数，再加上 bias。要将该方法直接扩展到多个 feature，可以按照公式 (1) 遍历每个元素，将其与对应参数相乘并累加，最后加上 bias。


In [ ]:
def predict_single_loop(x, w, b): 
    """
    single predict using linear regression
    
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters    
      b (scalar):  model parameter     
      
    Returns:
      p (scalar):  prediction
    """
    n = x.shape[0]
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]  
        p = p + p_i         
    p = p + b                
    return p

In [ ]:
# get a row from our training data
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

注意 `x_vec` 的 shape：它是一个包含 4 个元素的 1-D NumPy vector，shape 为 `(4,)`。结果 `f_wb` 是一个 scalar。


<a name="toc_15456_3.2"></a>
### 3.2 使用 Vector 计算单个预测值

公式 (1) 可以通过公式 (2) 中的 dot product 来实现，因此可以利用 vector 运算加快预测。

回顾 Python/NumPy 实验：NumPy 的 [`np.dot()`](https://numpy.org/doc/stable/reference/generated/numpy.dot.html) 可以计算两个 vector 的 dot product。


In [ ]:
def predict(x, w, b): 
    """
    single predict using linear regression
    Args:
      x (ndarray): Shape (n,) example with multiple features
      w (ndarray): Shape (n,) model parameters   
      b (scalar):             model parameter 
      
    Returns:
      p (scalar):  prediction
    """
    p = np.dot(x, w) + b     
    return p    

In [ ]:
# get a row from our training data
x_vec = X_train[0,:]
print(f"x_vec shape {x_vec.shape}, x_vec value: {x_vec}")

# make a prediction
f_wb = predict(x_vec,w_init, b_init)
print(f"f_wb shape {f_wb.shape}, prediction: {f_wb}")

结果及其 shape 与之前使用 loop 的版本相同。后续我们将使用 `np.dot` 执行此类运算。现在只需一条语句就能完成预测，因此大多数函数会直接包含该语句，而不再调用单独的预测函数。


<a name="toc_15456_4"></a>
## 4 计算多变量模型的 Cost

多变量模型的 cost function（代价函数）$J(\mathbf{w},b)$ 为：

$$
J(\mathbf{w},b)=\frac{1}{2m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)^2 \tag{3}
$$

其中：

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\mathbf{w}\cdot\mathbf{x}^{(i)}+b \tag{4}
$$

与之前的实验不同，这里的 $\mathbf{w}$ 和 $\mathbf{x}^{(i)}$ 都是 vector，从而能够支持多个 feature。


下面实现公式 (3) 和 (4)。这里采用*本课程常用的实现方式*：使用 `for` loop 遍历全部 `m` 个样本。


In [ ]:
def compute_cost(X, y, w, b): 
    """
    compute cost
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      cost (scalar): cost
    """
    m = X.shape[0]
    cost = 0.0
    for i in range(m):                                
        f_wb_i = np.dot(X[i], w) + b           #(n,)(n,) = scalar (see np.dot)
        cost = cost + (f_wb_i - y[i])**2       #scalar
    cost = cost / (2 * m)                      #scalar    
    return cost

In [ ]:
# Compute and display cost using our pre-chosen optimal parameters. 
cost = compute_cost(X_train, y_train, w_init, b_init)
print(f'Cost at optimal w : {cost}')

**预期结果：** 在给定的近似最优 `w` 下，cost 为 `1.5578904045996674e-12`。


<a name="toc_15456_5"></a>
## 5 多变量 Gradient Descent

多变量 gradient descent（梯度下降）重复执行以下更新，直到收敛：

$$
\begin{aligned}
w_j &:= w_j-\alpha\frac{\partial J(\mathbf{w},b)}{\partial w_j}, \qquad j=0,\ldots,n-1 \\
b &:= b-\alpha\frac{\partial J(\mathbf{w},b)}{\partial b}
\end{aligned}
\tag{5}
$$

其中，$n$ 是 feature 的数量。所有参数 $w_j$ 和 $b$ 必须**同时更新**，对应的 partial derivative（偏导数）为：

$$
\frac{\partial J(\mathbf{w},b)}{\partial w_j}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right)x_j^{(i)} \tag{6}
$$

$$
\frac{\partial J(\mathbf{w},b)}{\partial b}=\frac{1}{m}\sum_{i=0}^{m-1}\left(f_{\mathbf{w},b}(\mathbf{x}^{(i)})-y^{(i)}\right) \tag{7}
$$

- $m$ 是数据集中的训练样本数。
- $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ 是模型的预测值，$y^{(i)}$ 是 target。


<a name="toc_15456_5.1"></a>
### 5.1 计算多变量 Gradient

下面给出公式 (6) 和 (7) 的一种实现方式：

1. 外层 loop 遍历所有 $m$ 个样本。
2. 对每个样本，直接计算其对 $\frac{\partial J(\mathbf{w},b)}{\partial b}$ 的贡献并累加。
3. 内层 loop 遍历全部 $n$ 个 feature，计算该样本对每个 $\frac{\partial J(\mathbf{w},b)}{\partial w_j}$ 的贡献并累加。


In [ ]:
def compute_gradient(X, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      X (ndarray (m,n)): Data, m examples with n features
      y (ndarray (m,)) : target values
      w (ndarray (n,)) : model parameters  
      b (scalar)       : model parameter
      
    Returns:
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar):       The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape           #(number of examples, number of features)
    dj_dw = np.zeros((n,))
    dj_db = 0.

    for i in range(m):                             
        err = (np.dot(X[i], w) + b) - y[i]   
        for j in range(n):                         
            dj_dw[j] = dj_dw[j] + err * X[i, j]    
        dj_db = dj_db + err                        
    dj_dw = dj_dw / m                                
    dj_db = dj_db / m                                
        
    return dj_db, dj_dw

In [ ]:
#Compute and display gradient 
tmp_dj_db, tmp_dj_dw = compute_gradient(X_train, y_train, w_init, b_init)
print(f'dj_db at initial w,b: {tmp_dj_db}')
print(f'dj_dw at initial w,b: \n {tmp_dj_dw}')

**预期结果：**

- 初始 `w, b` 对应的 `dj_db`：`-1.6739251122999121e-06`。
- 初始 `w, b` 对应的 `dj_dw`：`[-2.73e-03 -6.27e-06 -2.22e-06 -6.92e-05]`。


<a name="toc_15456_5.2"></a>
### 5.2 多变量 Gradient Descent

下面的函数实现公式 (5)。


In [ ]:
def gradient_descent(X, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn w and b. Updates w and b by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      X (ndarray (m,n))   : Data, m examples with n features
      y (ndarray (m,))    : target values
      w_in (ndarray (n,)) : initial model parameters  
      b_in (scalar)       : initial model parameter
      cost_function       : function to compute cost
      gradient_function   : function to compute the gradient
      alpha (float)       : Learning rate
      num_iters (int)     : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,)) : Updated values of parameters 
      b (scalar)       : Updated value of parameter 
      """
    
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_db,dj_dw = gradient_function(X, y, w, b)   ##None

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               ##None
        b = b - alpha * dj_db               ##None
      
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( cost_function(X, y, w, b))

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]:8.2f}   ")
        
    return w, b, J_history #return final w,b and J history for graphing

在下一个代码单元中，你将测试该实现。


In [ ]:
# initialize parameters
initial_w = np.zeros_like(w_init)
initial_b = 0.
# some gradient descent settings
iterations = 1000
alpha = 5.0e-7
# run gradient descent 
w_final, b_final, J_hist = gradient_descent(X_train, y_train, initial_w, initial_b,
                                                    compute_cost, compute_gradient, 
                                                    alpha, iterations)
print(f"b,w found by gradient descent: {b_final:0.2f},{w_final} ")
m,_ = X_train.shape
for i in range(m):
    print(f"prediction: {np.dot(X_train[i], w_final) + b_final:0.2f}, target value: {y_train[i]}")

**预期结果：**

Gradient descent 得到的 `b` 为 `-0.00`，`w` 为 `[0.2 0. -0.01 -0.07]`。

| 预测值 | Target |
| --- | --- |
| 426.19 | 460 |
| 286.17 | 232 |
| 171.47 | 178 |


In [ ]:
# plot cost versus iteration  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12, 4))
ax1.plot(J_hist)
ax2.plot(100 + np.arange(len(J_hist[100:])), J_hist[100:])
ax1.set_title("Cost vs. iteration");  ax2.set_title("Cost vs. iteration (tail)")
ax1.set_ylabel('Cost')             ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('iteration step')   ;  ax2.set_xlabel('iteration step') 
plt.show()

*这些结果还不够理想！* Cost 仍在下降，预测也不够准确。下一个实验将探索如何改善这些结果。


<a name="toc_15456_6"></a>
## 6 恭喜完成实验！

在本实验中，你：

- 扩展了 linear regression 的相关函数，使其支持多个变量。
- 使用 NumPy 的 `np.dot` 实现了 vectorization。
